# 02 - Baseline Transformer + MIA (Alzheimer HF Dataset)

Objectif:
- Entrainer une baseline Transformer sur la nouvelle dataset preparee.
- Evaluer MIA standard et MIA robuste en mode black-box.

Prerequis:
- Executer d'abord le notebook de preparation dans `data_preparation_alzheimer_hf`.

In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

from research_protocol import evaluate_mia_research_protocol, set_seed
from research_protocol_robust import evaluate_mia_robust_protocol

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

def set_global_determinism(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

In [2]:
SEED = 42
ATTACK_SEEDS = [11, 22]

TARGET_DROPOUT_SAFE = 0.10
TARGET_DROPOUT_RISKY = 0.00
TARGET_EPOCHS_SAFE = 16
TARGET_EPOCHS_RISKY = 90

N_SHADOWS = 4
SHADOW_EPOCHS = 4
SHADOW_BATCH = 16

ROBUST_N_SHADOWS = 6
ROBUST_SHADOW_EPOCHS = 4
ROBUST_BOUNDARY_DIRS = 8
ROBUST_BOUNDARY_STEPS = 4
ROBUST_BOUNDARY_MAX_ALPHA = 2.0

ATTACK_MIN_AUC_TARGET = 0.55
TARGET_MIN_TEST_ACC = 0.72
MIN_TRAIN_SAMPLES = 12000

# Keep this True for full benchmark; set False if you only want quick accuracy check.
RUN_ATTACKS = True

set_global_determinism(SEED)

prepared_path = os.path.join('..', 'data_preparation_alzheimer_hf', 'prepared_alzheimer_hf_transformer.npz')
if not os.path.exists(prepared_path):
    raise FileNotFoundError('prepared_alzheimer_hf_transformer.npz not found. Run preparation notebook first.')

b = np.load(prepared_path, allow_pickle=True)
X = b['X'].astype(np.float32)
y = b['y'].astype(np.int32)
X_train = b['X_train'].astype(np.float32)
X_test = b['X_test'].astype(np.float32)
y_train = b['y_train'].astype(np.int32)
y_test = b['y_test'].astype(np.int32)
X_shadow_raw = b['X_shadow_raw'].astype(np.float32)
y_shadow = b['y_shadow'].astype(np.int32)

# Re-split target pool to increase train size and improve model utility.
X_target_pool = np.vstack([X_train, X_test]).astype(np.float32)
y_target_pool = np.concatenate([y_train, y_test]).astype(np.int32)
X_train, X_test, y_train, y_test = train_test_split(
    X_target_pool,
    y_target_pool,
    test_size=0.40,
    random_state=SEED + 123,
    stratify=y_target_pool,
)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1]).astype(np.float32)
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1]).astype(np.float32)

if X_train_seq.shape[0] < MIN_TRAIN_SAMPLES:
    raise ValueError(f'Train size too small: {X_train_seq.shape[0]} < {MIN_TRAIN_SAMPLES}.')

print(f'Loaded: X_train={X_train_seq.shape}, X_test={X_test_seq.shape}, X_shadow={X_shadow_raw.shape}')
print(f'Class ratio train={y_train.mean():.4f}, test={y_test.mean():.4f}')

def build_transformer(n_features, dropout=0.15):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(256, activation='relu')(inp)
    x = layers.Dropout(dropout)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(8e-4),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

Loaded: X_train=(31185, 1, 7), X_test=(20791, 1, 7), X_shadow=(22276, 7)
Class ratio train=0.4135, test=0.4134


In [3]:
print('=== Baseline utility model (safe) ===')
set_seed(SEED + 1)
tf.keras.backend.clear_session()

model_safe = build_transformer(n_features=X_train.shape[1], dropout=TARGET_DROPOUT_SAFE)
es = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=0)
model_safe.fit(
    X_train_seq,
    y_train,
    epochs=TARGET_EPOCHS_SAFE,
    batch_size=64,
    validation_split=0.2,
    callbacks=[es],
    verbose=0,
)

p_tr_safe = model_safe.predict(X_train_seq, verbose=0).ravel()
p_te_safe = model_safe.predict(X_test_seq, verbose=0).ravel()

safe_train_acc = accuracy_score(y_train, (p_tr_safe >= 0.5).astype(int))
safe_test_acc = accuracy_score(y_test, (p_te_safe >= 0.5).astype(int))
print(pd.DataFrame([{
    'model': 'safe_reference',
    'train_acc': safe_train_acc,
    'test_acc': safe_test_acc,
    'gap': safe_train_acc - safe_test_acc,
    'train_auc': roc_auc_score(y_train, p_tr_safe),
    'test_auc': roc_auc_score(y_test, p_te_safe),
}]).round(4))

print('\n=== Baseline attack target (risky) ===')
set_seed(SEED + 2)
tf.keras.backend.clear_session()

# Intentionally no early stopping on risky model to encourage mild overfitting.
model_attack = build_transformer(n_features=X_train.shape[1], dropout=TARGET_DROPOUT_RISKY)
model_attack.fit(
    X_train_seq,
    y_train,
    epochs=TARGET_EPOCHS_RISKY,
    batch_size=64,
    verbose=0,
)

p_tr_attack = model_attack.predict(X_train_seq, verbose=0).ravel()
p_te_attack = model_attack.predict(X_test_seq, verbose=0).ravel()

attack_train_acc = accuracy_score(y_train, (p_tr_attack >= 0.5).astype(int))
attack_test_acc = accuracy_score(y_test, (p_te_attack >= 0.5).astype(int))
print(pd.DataFrame([{
    'model': 'attack_target',
    'train_acc': attack_train_acc,
    'test_acc': attack_test_acc,
    'gap': attack_train_acc - attack_test_acc,
    'train_auc': roc_auc_score(y_train, p_tr_attack),
    'test_auc': roc_auc_score(y_test, p_te_attack),
}]).round(4))

if attack_test_acc >= TARGET_MIN_TEST_ACC:
    print(f"\nOK: test_acc={attack_test_acc:.4f} >= target {TARGET_MIN_TEST_ACC:.2f}")
else:
    print(f"\nWARNING: test_acc={attack_test_acc:.4f} < target {TARGET_MIN_TEST_ACC:.2f}")

if not RUN_ATTACKS:
    print('\nRUN_ATTACKS=False -> skip MIA attacks for quick utility check.')
else:
    print('\n=== MIA standard (strong attacker) ===')
    baseline_summary, baseline_per_seed = evaluate_mia_research_protocol(
        target_model=model_attack,
        X_train_seq=X_train_seq,
        y_train=y_train,
        X_test_seq=X_test_seq,
        y_test=y_test,
        X_shadow_raw=X_shadow_raw,
        y_shadow=y_shadow,
        model_builder=lambda nf: build_transformer(n_features=nf, dropout=TARGET_DROPOUT_RISKY),
        seed_list=ATTACK_SEEDS,
        n_shadows=N_SHADOWS,
        shadow_epochs=SHADOW_EPOCHS,
        shadow_batch_size=SHADOW_BATCH,
    )
    display(baseline_summary.round(4))

    print('\n=== MIA robuste (black-box) ===')
    baseline_robust_summary, baseline_robust_per_seed = evaluate_mia_robust_protocol(
        target_model=model_attack,
        x_train_seq=X_train_seq,
        y_train=y_train,
        x_test_seq=X_test_seq,
        y_test=y_test,
        x_shadow_raw=X_shadow_raw,
        y_shadow=y_shadow,
        model_builder=lambda nf: build_transformer(n_features=nf, dropout=TARGET_DROPOUT_RISKY),
        seed_list=ATTACK_SEEDS,
        n_shadows=ROBUST_N_SHADOWS,
        shadow_epochs=ROBUST_SHADOW_EPOCHS,
        shadow_batch_size=SHADOW_BATCH,
        boundary_dirs=ROBUST_BOUNDARY_DIRS,
        boundary_steps=ROBUST_BOUNDARY_STEPS,
        boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
    )
    display(baseline_robust_summary.round(4))

    print('\nQuick view (AUC):')
    quick_standard = baseline_summary[['attack', 'auc_mean', 'auc_std']].sort_values('auc_mean', ascending=False)
    display(quick_standard.round(4))
    quick_robust = baseline_robust_per_seed[['seed', 'auc', 'selected_scorer', 'tpr_at_1pct_fpr', 'tpr_at_5pct_fpr']]
    display(quick_robust.round(4))

    std_auc = float(quick_standard.iloc[0]['auc_mean'])
    rob_auc = float(baseline_robust_summary.iloc[0]['auc_mean'])
    if std_auc < ATTACK_MIN_AUC_TARGET and rob_auc < ATTACK_MIN_AUC_TARGET:
        print(f'\nWARNING: attaques faibles (std={std_auc:.4f}, robust={rob_auc:.4f}).')
        print('Augmenter N_SHADOWS/epochs ou reduire la taille train pour un benchmark plus agressif.')

=== Baseline utility model (safe) ===

            model  train_acc  test_acc     gap  train_auc  test_auc
0  safe_reference      0.724    0.7178  0.0063     0.7961    0.7871

=== Baseline attack target (risky) ===
           model  train_acc  test_acc     gap  train_auc  test_auc
0  attack_target     0.7577    0.6803  0.0774     0.8371    0.7351


=== MIA standard (strong attacker) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
2,threshold_loss,0.5439,0.0040,0.5759,0.0045,0.6247,0.0002,0.7341,0.0193,0.6749,0.0081
0,logistic,0.5427,0.0020,0.6098,0.0007,0.6118,0.0008,0.9564,0.0026,0.7463,0.0002
1,shadow_meta,0.5248,0.0023,0.5386,0.0029,0.6143,0.0023,0.6203,0.0026,0.6173,0.0025



=== MIA robuste (black-box) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,shadow_meta,0.536,0.0037,0.5234,0.0018,0.6234,0.0018,0.5196,0.0015,0.5668,0.0016



Quick view (AUC):


,attack,auc_mean,auc_std
2,threshold_loss,0.5439,0.0040
0,logistic,0.5427,0.0020
1,shadow_meta,0.5248,0.0023


,seed,auc,selected_scorer,tpr_at_1pct_fpr,tpr_at_5pct_fpr
0,11,0.5386,meta_only,0.0105,0.0613
1,22,0.5334,meta_only,0.0111,0.0572



Augmenter N_SHADOWS/epochs ou reduire la taille train pour un benchmark plus agressif.
